In [ ]:
import polars as pl
import numpy as np

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
df = pl.read_csv("../data/stockdata3.csv")

In [ ]:
stocks = [c for c in df.columns if c != "day" and c != "timestr"]
returns = [f"r{s}" for s in stocks]

In [ ]:
INTRA = 0
EXTRA = 1
CLOSE = 2


def get_ret(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .select(
            [
                pl.col("day"),
                pl.col("datetime"),
                ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}"),
            ]
        )
        .with_columns(
            [
                pl.when(pl.col("day").diff() == 0)
                .then(pl.lit(INTRA))
                .otherwise(pl.lit(EXTRA))
                .alias("return_type")
            ]
        )
    )


def get_ret_cc(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .group_by("day")
        .agg([pl.col(s).drop_nulls().last()])
        .sort("day")  # must sort before shift(1), group_by output is unordered
        .with_columns(
            [
                ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}"),
                pl.lit(CLOSE).alias("return_type"),
            ]
        )
    )


def get_ret_all(df, s):
    ret = get_ret(df, s[0])
    for stock in s[1:]:
        print(stock)
        ret = ret.join(get_ret(df, stock), on="datetime", how="inner")

    return ret


# def add_ret(df):
#     return (
#         df.sort("datetime")
#         .fill_nan(None)
#         .with_columns(
#             [
#                 ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}")
#                 for s in stocks
#             ]
#         )
#         .with_columns(
#             [
#                 pl.when(pl.col("day").diff() == 0)
#                 .then(pl.lit(INTRA))
#                 .otherwise(pl.lit(EXTRA))
#                 .alias("return_type")
#             ]
#         )
#     )

In [ ]:
def daily_vol_from_ret(df, returns):
    if isinstance(returns, str):
        return df.group_by("day").agg(pl.col(returns).pow(2).sum().sqrt())
    else:
        return df.group_by("day").agg(
            [
                (pl.col(s1) * pl.col(s2)).sum().sqrt().alias(f"")
                for s1 in returns
                for s2 in returns
            ]
        )


def daily_vol(df, stocks, return_type=None):

    mask = (
        pl.col("return_type") == return_type
        if return_type is not None
        else pl.lit(True)
    )
    if isinstance(stocks, str):
        stocks = [stocks]
    returns = [f"r{s}" for s in stocks]
    vols = daily_vol_from_ret(get_ret(df, stocks[0]).filter(mask), returns[0])
    for s, r in zip(stocks[1:], returns[1:]):
        vols = vols.join(daily_vol_from_ret(get_ret(df, s).filter(mask), r), on="day")
    return vols


def combine_intraday_gap_vol(intraday_vols, gap_vols, returns, day_col="day"):
    """Combine intraday RV and overnight gap RV on the daily-vol scale."""
    joined = intraday_vols.join(
        gap_vols.select(day_col, *returns), on=day_col, how="left", suffix="_gap"
    )
    return joined.with_columns(
        [
            (pl.col(r).pow(2) + pl.col(f"{r}_gap").fill_null(0).pow(2)).sqrt().alias(r)
            for r in returns
        ]
    ).select(day_col, *returns)

In [ ]:
# deal with c: likely split
def normalize_c(df):
    idxs = df.filter(pl.col("c").diff().abs() > 0.4 * pl.col("c"))["index"]
    if len(idxs) > 0:
        index = idxs[0]
    else:
        return df
    print(index)
    return df.with_columns(
        pl.when(pl.col("index") < index)
        .then(pl.col("c") / 2)
        .otherwise(pl.col("c"))
        .alias("c")
    )

In [ ]:
# deal with c: likely split
# deal with c: likely split
def mask_micro_noise(col, thd=100, neighbor=1):
    ratio = thd / 1e4  # bps
    ret = (pl.col(col) / pl.col(col).shift(1)).log()
    mask = pl.lit(False)
    for n in range(1, neighbor + 1):
        mask |= (
            (ret * ret.shift(-n) < 0)
            & (ret.abs() > ratio)
            & (ret.shift(-n).abs() > ratio)
        )
    return mask


def mask_denoised(col, thd=100, neighbor=1):
    return ~mask_micro_noise(col, thd, neighbor)


# def clean_d(df):
#     return df

In [ ]:
def downsample(df, freq, keep_open=True):
    sample_time = "__sample_datetime"
    sampled = (
        df.with_columns(pl.col("datetime").alias(sample_time))
        .group_by_dynamic("datetime", every=freq, group_by="day")
        .agg(
            pl.exclude(["day", "datetime", sample_time]).last(),
            pl.col(sample_time).last(),
        )
        .drop("datetime")
        .rename({sample_time: "datetime"})
    )
    if keep_open:
        opens = df.sort("datetime").group_by("day", maintain_order=True).head(1)
        sampled = pl.concat([opens, sampled], how="diagonal_relaxed").unique(
            ["day", "datetime"], keep="first"
        )
    return sampled.sort(["day", "datetime"])

## load data

In [ ]:
df = (
    df.with_columns(
        (pl.date(2000, 1, 1) + pl.duration(days=pl.col("day") - 1))
        .dt.combine(pl.col("timestr").str.to_time("%H:%M:%S"))
        .alias("datetime")
    )
    if "datetime" not in df.columns
    else df
)
display(df.select("day", "timestr", "datetime"))
df = df.with_row_index() if "index" not in df.columns else df

In [ ]:
# clean a and d
df = df.with_columns(
    pl.when(pl.col("a") == 0).then(None).otherwise(pl.col("a")).alias("a"),
    pl.when(pl.col("d") == 1).then(None).otherwise(pl.col("d")).alias("d"),
)
# clean c
df = normalize_c(df)
# filter d
NOISE_THD = 100  # 100 bps
# df = df.filter(mask_denoised("d", NOISE_THD))

In [ ]:
# categories:
stocks_normal = ["a", "c", "e"]  # leave it as is
stocks_downsample_5min_denoise = ["d"]  # down sample to 5min
stocks_downsample_120min = ["b"]  # down sample to 120min
stocks_jump = ["f"]  # special treat, do Poisson jump or monthly close-close volatility

In [ ]:
display(downsample(df, "10m"))

In [ ]:
df

In [ ]:
vols = daily_vol(df, stocks_normal + stocks_jump, return_type=INTRA)
vols = vols.join(
    daily_vol(
        downsample(df.filter(mask_denoised("d", NOISE_THD)), "5m"),
        stocks_downsample_5min_denoise,
        return_type=INTRA,
    ),
    on="day",
)
vols = vols.join(
    daily_vol(downsample(df, "120m"), stocks_downsample_120min, return_type=INTRA),
    on="day",
)

vols_intraday = vols.sort("day")
vols_gap = daily_vol(df, stocks, return_type=EXTRA).sort("day")
vols = combine_intraday_gap_vol(vols_intraday, vols_gap, returns).sort("day")

# close-to-close daily returns, one row per day --- join on day (not concat:
# each get_ret_cc frame is 1-row-per-day, a horizontal concat would misalign).
ret_cc = get_ret_cc(df, stocks[0]).select("day", f"r{stocks[0]}")
for s in stocks[1:]:
    ret_cc = ret_cc.join(get_ret_cc(df, s).select("day", f"r{s}"), on="day")
ret_cc = ret_cc.sort("day")

In [ ]:
ret_cc

In [ ]:
# vols_bm = daily_vol(df, stocks_normal + stocks_jump)
# vols_bm = vols_bm.join(daily_vol(df, stocks_downsample), on="day")

In [ ]:
vols = vols.sort("day")
# vols_bm = vols_bm.sort("day")

In [ ]:
for v in returns:
    # print(f"=============={v}==============")
    plt.step(vols["day"], vols[v], label=v)
    # plt.plot(vols_bm[v])
plt.legend()
plt.show()

## Check autocorrelation

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

acfs, pacfs = {}, {}
for r in returns:
    data = vols[r].drop_nulls().to_numpy()
    acfs[r] = acf(data, nlags=40, alpha=0.05)
    pacfs[r] = pacf(data, nlags=40, alpha=0.05)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    plot_acf(data, lags=40, alpha=0.05, ax=ax1)
    plot_pacf(data, lags=40, alpha=0.05, ax=ax2)
    ax1.set_title(f"ACF {r}")
    ax2.set_title(f"PACF {r}")
    plt.tight_layout()
    plt.show()

## Check daily volatilities

In [ ]:
for r in returns:
    print(f"====================={r}=====================")
    print(vols[r].mean(), vols[r].std())
    plt.step(vols["day"], vols[r])
    plt.title(r)
    plt.show()

## Check the short day

In [ ]:
print(vols.filter(pl.col("day") == 327))
print(vols.mean())

In [ ]:
np.sqrt(391 / 211)

## Check lagged cross correlation

In [ ]:
for lag in [1, 5, 21]:
    lagged = (
        vols.select(returns).shift(lag).rename({c: f"{c}_lag{lag}" for c in returns})
    )
    combined = pl.concat([vols.select(returns), lagged], how="horizontal").drop_nulls()
    corr = combined.corr()
    n = len(returns)
    block = corr[:n, n:]
    block = block.rename({f"{c}_lag{lag}": c for c in returns})
    block = block.insert_column(0, pl.Series("name", returns))
    print(f"\ncorr(RV_i(t), RV_j(t-{lag})):")
    print(block)

## Model fitting/prediction

In [ ]:
from collections.abc import Callable, Sequence


# ---- pooling: aggregate the daily-RV expr over a window of w trading days ----
# Daily RV is itself sqrt(daily variance); the choice is how those vols compose.
def pool_mean(e: pl.Expr, w: int) -> pl.Expr:
    """Plain average of daily RV over w days --- the literal Corsi HAR pooling."""
    return e.rolling_mean(w)


def pool_rms(e: pl.Expr, w: int) -> pl.Expr:
    """RMS of daily RV over w days: sqrt(mean RV^2). Dimensionally consistent
    (variances are additive, vols are not); stays on the daily-vol scale."""
    return e.pow(2).rolling_mean(w).sqrt()


# ---- feature builders: act on the daily-RV expr, return a causal (info<=t) expr ----
def ewma(half_life: float, on_variance: bool = False):
    """Exponentially weighted moving avg of daily RV, info up to row t (causal).

    A self-contained causal feature --- pass it in `feature_fns`.
    half_life : decay in trading days. RiskMetrics lambda=0.94 ~= half_life 11.
    on_variance : if True, EWMA the variance (RV^2) RiskMetrics-style and sqrt
                  back; if False, EWMA the vol series directly.
    """

    def fn(e: pl.Expr) -> pl.Expr:
        if on_variance:
            return e.pow(2).ewm_mean(half_life=half_life, adjust=False).sqrt()
        return e.ewm_mean(half_life=half_life, adjust=False)

    return fn


def make_features_targets(
    vols: pl.DataFrame,
    cols: Sequence[str],
    windows: Sequence[int] = (1, 5, 21),
    horizon: int = 21,
    pool: Callable[[pl.Expr, int], pl.Expr] = pool_rms,
    feature_fns: dict[str, Callable[[pl.Expr], pl.Expr]] | None = None,
    target_fn: Callable[[pl.Expr], pl.Expr] | None = None,
    day_col: str = "day",
    drop_nulls: bool = False,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Turn a daily realized-vol frame into (features, target) frames.

    `vols` has one row per trading day plus `day_col`. Rows are assumed to be
    consecutive trading days, so a window / horizon of N rows == N trading days.
    All features use info up to and including day t; the target is strictly
    forward (t+1..t+h), so there is no overlap and no leakage. For row t:
      HAR feature {c}_avg{w} = pool of daily RV over the trailing w days [t-w+1..t]
      extra       {c}{name}  = feature_fns[name](col c)   (causal)
      target      {c}_fwd    = pool of daily RV over the forward h days [t+1..t+h]

    windows : HAR averaging windows in trading days. (1,5,21) are the classic
              daily / weekly / monthly HAR-RV components (Corsi 2009 uses 22).
              w=1 is RV(t) itself and keeps the bare name {c}.
    horizon : 21 is the usual "1 month" (252 trading days / 12 ~= 21); Corsi's
              HAR-RV uses 22; 20 is just a round "4 weeks". Default 21.
    pool    : (expr, w)->expr aggregator used for BOTH the HAR features and the
              default target, so the two always share one convention. pool_rms
              (default) is dimensionally consistent; pool_mean is literal Corsi.
    feature_fns : optional {name: fn(expr)->expr} extra causal features, e.g.
              {"_ewma11": ewma(11)}. Applied per column, info up to t.
    target_fn : optional override of the forward target. Default is
              pool(., horizon) shifted back h rows. Must be forward-looking.
    drop_nulls : if False (default) keep every row from day 1 --- partial
              windows (head) and missing forward targets (tail) stay null, so
              you can experiment with models that don't need a full HAR window.
              Set True for a ready-to-fit frame with the head/tail trimmed.

    Returns (X, y), both keyed by `day_col` and row-aligned.
    """
    vols = vols.sort(day_col)
    target_fn = target_fn or (lambda e: pool(e, horizon).shift(-horizon))

    feature_exprs = [
        pool(pl.col(c), w).alias(f"{c}" + (f"_avg{w}" if w != 1 else ""))
        for c in cols
        for w in windows
    ]
    if feature_fns:
        feature_exprs += [
            fn(pl.col(c)).alias(f"{c}{name}")
            for name, fn in feature_fns.items()
            for c in cols
        ]
    target_exprs = [target_fn(pl.col(c)).alias(f"{c}_fwd") for c in cols]

    combined = vols.select(pl.col(day_col), *feature_exprs, *target_exprs)
    if drop_nulls:
        combined = combined.drop_nulls()  # trims the window head + horizon tail

    feat_names = [e.meta.output_name() for e in feature_exprs]
    tgt_names = [e.meta.output_name() for e in target_exprs]
    return combined.select(day_col, *feat_names), combined.select(day_col, *tgt_names)

In [ ]:
X, y = make_features_targets(
    vols,
    returns,
    # 5 trading days ~= one week; 10/15 are two-/three-week windows.
    windows=[1, 5, 10, 11, 15, 21],
    horizon=21,
    pool=pool_rms,  # pool_mean for the literal Corsi HAR average
    feature_fns={
        "_ewma1": ewma(1),
        "_ewma5": ewma(5),
        "_ewma10": ewma(10),
        "_ewma11": ewma(11),
        "_ewma15": ewma(15),
        "_ewma21": ewma(21),
        "_ewma44": ewma(44),
    },
)
print("X:", X.shape, " y:", y.shape)
display(X.head())
display(y.head())

### PCA diagnostic on the pre-modeling feature matrix

This checks the standardized HAR/EWMA feature matrix immediately after `X, y` are built, before adding the GARCH benchmark or any model-specific transforms.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_feature_cols = [c for c in X.columns if c != "day"]
X_pca = X.select(["day", *pca_feature_cols]).drop_nulls().drop_nans()
X_pca_scaled = StandardScaler().fit_transform(X_pca.select(pca_feature_cols).to_numpy())

pca = PCA()
pca_scores = pca.fit_transform(X_pca_scaled)
pca_summary = pl.DataFrame(
    {
        "component": [f"PC{i + 1}" for i in range(len(pca.explained_variance_ratio_))],
        "explained_variance": pca.explained_variance_,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_),
    }
)
display(pca_summary.head(10))

n_loading_components = min(5, len(pca_feature_cols))
loading_cols = [f"PC{i + 1}" for i in range(n_loading_components)]
pca_loadings = pl.DataFrame(
    {
        "feature": pca_feature_cols,
        **{f"PC{i + 1}": pca.components_[i] for i in range(n_loading_components)},
    }
)
display(
    pca_loadings.with_columns(
        pl.max_horizontal([pl.col(c).abs() for c in loading_cols]).alias(
            "max_abs_loading"
        )
    )
    .sort("max_abs_loading", descending=True)
    .head(15)
)

plt.figure(figsize=(7, 4))
plt.plot(
    pca_summary["component"][:10],
    pca_summary["cumulative_ratio"][:10],
    marker="o",
)
plt.ylim(0, 1.02)
plt.ylabel("cumulative explained variance")
plt.xlabel("principal component")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
X.sort("day")

### GARCH benchmark

In [ ]:
from arch import arch_model


def fit_garch(returns, p=1, q=1):
    model = arch_model(returns, vol="Garch", p=p, q=q, rescale=True)
    res = model.fit(disp="off")
    print(res.summary())
    return res


def garch_predict(res):
    forecast = res.forecast(horizon=21, reindex=False)
    # Sum daily variances over 21 days for monthly variance
    monthly_var = forecast.variance.iloc[-1].sum()
    return np.sqrt(monthly_var)
    # monthly_vol_ann = np.sqrt(monthly_var * 252 / 21) / 100  # if rescaled to pct
    # print(f"Annualized vol: {monthly_vol_ann:.2%}")

In [ ]:
def garch_features(ret_cc, stocks, horizon=21, day_col="day", min_obs=63):
    """Causal expanding-window GARCH(1,1) features from close-to-close returns.

    For each day t, fit only on close-to-close returns observed through t and
    forecast the average daily volatility over t+1..t+horizon. This makes the
    GARCH benchmark comparable to the causal HAR/EWMA features in CV.
    """
    ret_cc = ret_cc.sort(day_col)
    out = {day_col: ret_cc[day_col]}
    for s in stocks:
        col = f"r{s}"
        r = ret_cc[col].to_numpy()
        filtered = np.full(len(r), np.nan)
        fwd = np.full(len(r), np.nan)
        for i in range(len(r)):
            hist = r[: i + 1]
            hist = hist[np.isfinite(hist)]
            if len(hist) < min_obs:
                continue
            res = arch_model(hist, vol="Garch", p=1, q=1, rescale=True).fit(
                disp="off", show_warning=False
            )
            sc = res.scale  # arch may rescale returns internally; undo it
            fc = res.forecast(horizon=horizon, reindex=False)
            filtered[i] = np.asarray(res.conditional_volatility)[-1] / sc
            fwd[i] = np.sqrt(fc.variance.to_numpy()[-1].mean()) / sc
        out[f"r{s}_garch"] = filtered
        out[f"r{s}_garch_fwd"] = fwd
    return pl.DataFrame(out).with_columns(pl.exclude(day_col).fill_nan(None))

In [ ]:
garch = garch_features(ret_cc, stocks, horizon=21)
display(garch.head())

# join GARCH columns onto X (drop stale columns so re-running stays idempotent)
garch_cols = [c for c in X.columns if c.endswith("_garch") or c.endswith("_garch_fwd")]
if garch_cols:
    X = X.drop(garch_cols)
X = X.join(garch, on="day", how="left")
print("X with GARCH:", X.shape)

### check rolling monthly vol

In [ ]:
# for c in X.columns:
#     if c == "day":
#         continue
#     c1 = c.split("_")[0] + "_fwd"
#     print(f"=============={c}, {c1}===============")
#     plt.scatter(X[c], y[c1])
#     print(np.corrcoef(X[c], y[c1])[0, 1])
#     plt.show()

vol_log_avg = sum([X[f"log_{r}_avg21"] / 6 for r in returns])
for r in returns:
    plt.plot(y["day"], y[f"{r}_fwd"])
    plt.twinx()
    plt.plot(X["day"], vol_log_avg, label="log ave", color="orange")
    plt.plot(X["day"], X[f"log_{r}_avg15"], label="11", color="red")
    plt.legend()
    plt.show()

In [ ]:
import itertools
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.linear_model import LinearRegression


class PurgedWalkForwardCV:
    """Walk-forward CV for forward-looking targets.

    Train always precedes test; between them the last `purge` train rows are
    dropped --- their target window (horizon trading days) reaches into the
    test block, so keeping them would train the model on test-period outcomes.
    Because train is always before test there is no post-test embargo to do:
    purging the train tail removes the only leak path. Rows must be sorted by
    day, one consecutive trading day each.

    n_splits   : number of sequential test folds.
    purge      : rows dropped from the train tail; set to the target horizon.
    min_train  : rows before the first test fold.
    train_span : None -> expanding train window; int -> rolling window size.
    """

    def __init__(self, n_splits=5, purge=21, min_train=63, train_span=None):
        self.n_splits = n_splits
        self.purge = purge
        self.min_train = min_train
        self.train_span = train_span

    def split(self, X, y=None, groups=None):
        n = len(X)
        fold = (n - self.min_train) // self.n_splits
        if fold <= 0:
            raise ValueError("not enough rows for given n_splits / min_train")
        for k in range(self.n_splits):
            test_start = self.min_train + k * fold
            test_end = n if k == self.n_splits - 1 else test_start + fold
            train_end = test_start - self.purge
            if train_end <= 0:
                continue
            lo = 0 if self.train_span is None else max(0, train_end - self.train_span)
            yield np.arange(lo, train_end), np.arange(test_start, test_end)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


class PassthroughRegressor(BaseEstimator, RegressorMixin):
    """Dummy model: returns one feature column verbatim as the prediction.

    No fitting --- use it to score a precomputed forecast inside the same CV
    pipeline as the trainable models. To benchmark GARCH, call evaluate_model
    with features=["r{s}_garch_fwd"] and model=PassthroughRegressor().

    col : index of the feature column to pass through (0 = first feature).
    """

    def __init__(self, col=0):
        self.col = col

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.asarray(X)[:, self.col]


class LogLogLinearRegressor(BaseEstimator, RegressorMixin):
    """Linear regression in log(feature) / log(target), predicted on original scale."""

    def __init__(self, fit_intercept=True, eps=1e-12):
        self.fit_intercept = fit_intercept
        self.eps = eps

    def _log_positive(self, values):
        values = np.asarray(values, dtype=float)
        return np.log(np.maximum(values, self.eps))

    def fit(self, X, y):
        self.model_ = LinearRegression(fit_intercept=self.fit_intercept)
        self.model_.fit(self._log_positive(X), self._log_positive(y).ravel())
        self.coef_ = self.model_.coef_
        self.intercept_ = self.model_.intercept_
        return self

    def predict(self, X):
        return np.exp(self.model_.predict(self._log_positive(X)))


class ForwardStagewiseRegressor(BaseEstimator, RegressorMixin):
    """Forward stagewise residual regression with fixed coefficients.

    Features are normalized on each fit. At each step, regress the current
    residual on every remaining normalized feature individually, select the
    largest positive slope, fix that slope, and update the residual. Stop when
    the best remaining slope is at or below min_coef.
    """

    def __init__(self, max_features=8, min_coef=0.0, feature_names=None):
        self.max_features = max_features
        self.min_coef = min_coef
        self.feature_names = feature_names

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        n_features = X.shape[1]
        names = (
            list(self.feature_names)
            if self.feature_names is not None
            else [f"x{i}" for i in range(n_features)]
        )

        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.scale_ = np.where(self.scale_ == 0, 1.0, self.scale_)
        Xn = (X - self.mean_) / self.scale_

        selected = []
        remaining = list(range(n_features))
        intercept_norm = float(y.mean())
        residual = y - intercept_norm
        coef_norm = np.zeros(n_features)
        self.selection_path_ = []
        max_features = (
            n_features
            if self.max_features is None
            else min(self.max_features, n_features)
        )

        for step in range(1, max_features + 1):
            if not remaining:
                break

            X_remaining = Xn[:, remaining]
            denom = np.sum(X_remaining**2, axis=0)
            slopes = np.full(len(remaining), -np.inf)
            valid = denom > 0
            slopes[valid] = (X_remaining[:, valid].T @ residual) / denom[valid]

            best_pos = int(np.argmax(slopes))
            best_slope = float(slopes[best_pos])
            if best_slope <= self.min_coef:
                break

            feature_idx = remaining.pop(best_pos)
            selected.append(feature_idx)
            coef_norm[feature_idx] = best_slope
            residual = residual - best_slope * Xn[:, feature_idx]
            self.selection_path_.append(
                {
                    "step": step,
                    "feature": names[feature_idx],
                    "coef_norm": best_slope,
                }
            )

        coef = coef_norm / self.scale_
        intercept = float(intercept_norm - np.dot(coef_norm, self.mean_ / self.scale_))

        self.coef_ = coef
        self.intercept_ = intercept
        self.coef_norm_ = coef_norm
        self.selected_idx_ = np.array(selected, dtype=int)
        self.selected_feature_names_ = [names[i] for i in selected]
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.coef_ + self.intercept_


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def mse(y_true, y_pred):
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


def qlike(y_true, y_pred, eps=1e-6):
    """QLIKE on variance forecasts; negative vol forecasts get near-zero variance."""
    true_vol = np.asarray(y_true, dtype=float)
    pred_vol = np.asarray(y_pred, dtype=float)
    true_var = np.maximum(true_vol, 0.0) ** 2 + eps
    pred_var = np.maximum(pred_vol, 0.0) ** 2 + eps
    ratio = true_var / pred_var
    return float(np.mean(ratio - np.log(ratio) - 1.0))


DEFAULT_LOSS_FNS = {"rmse": rmse, "mse": mse, "qlike": qlike}


def _normalize_loss_fns(loss_fns=None, scorer=None):
    if loss_fns is None:
        loss_fns = scorer if scorer is not None else DEFAULT_LOSS_FNS
    if isinstance(loss_fns, dict):
        return dict(loss_fns)
    if callable(loss_fns):
        name = getattr(loss_fns, "__name__", "loss")
        return {name: loss_fns}
    normalized = {}
    for i, fn in enumerate(loss_fns):
        base_name = getattr(fn, "__name__", f"loss_{i + 1}")
        name = base_name if base_name not in normalized else f"{base_name}_{i + 1}"
        normalized[name] = fn
    return normalized


def evaluate_model(
    X,
    y,
    model,
    features,
    target,
    param_grid=None,
    cv=None,
    horizon=21,
    day_col="day",
    loss_fns=None,
    primary_loss="rmse",
    scorer=None,
):
    """Purged walk-forward CV for one (features -> target) model.

    X, y    : day-keyed frames (make_features_targets output, + GARCH join);
              joined on `day_col`, sorted, null rows dropped.
    model   : an unfitted sklearn-style estimator (must be clone-able).
    features: feature column names in X the model is given.
    target  : target column name in y.
    param_grid : optional {param: [values]}; every combination is scored and
              the one with the lowest mean fold error wins.
    cv      : splitter; default PurgedWalkForwardCV(purge=horizon).
    loss_fns: dict/list/callable of (y_true, y_pred) -> loss; default
              {rmse, mse, qlike}. A single legacy `scorer` is still accepted.
    primary_loss : loss name used for model/parameter selection.

    Returns dict for the BEST param combo with:
      best_params, best_model (refit on all rows),
      <loss>_mean/std, train_<loss>_mean/std --- validation/training stats,
      cv_mean, train_cv_mean, fold_errors    --- aliases for primary_loss,
      r2, pred_mean, pred_std,               --- prediction stats (pooled OOF),
      true_mean, true_std,                   --- target stats (pooled OOF),
      results (one row per param combo, sorted by error; also includes the
               per-combo r2 / pred / true stats).
    """
    loss_fns = _normalize_loss_fns(loss_fns, scorer=scorer)
    if primary_loss not in loss_fns:
        primary_loss = next(iter(loss_fns))
    primary_mean_col = f"{primary_loss}_mean"
    primary_std_col = f"{primary_loss}_std"
    cv = cv or PurgedWalkForwardCV(purge=horizon)
    d = (
        X.select([day_col, *features])
        .join(y.select([day_col, target]), on=day_col)
        .sort(day_col)
        .drop_nulls()
        .drop_nans()
    )
    Xa, ya = d.select(features).to_numpy(), d.select(target).to_numpy().ravel()

    grid = param_grid or {}
    combos = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]

    rows = []
    for params in combos:
        errs = {name: [] for name in loss_fns}
        train_errs = {name: [] for name in loss_fns}
        preds, trues = [], []
        for tr, te in cv.split(Xa):
            m = clone(model).set_params(**params)
            m.fit(Xa[tr], ya[tr])
            train_p = m.predict(Xa[tr])
            p = m.predict(Xa[te])
            for loss_name, loss_fn in loss_fns.items():
                train_errs[loss_name].append(loss_fn(ya[tr], train_p))
                errs[loss_name].append(loss_fn(ya[te], p))
            preds.append(p)
            trues.append(ya[te])
        errs = {name: np.asarray(values, dtype=float) for name, values in errs.items()}
        train_errs = {
            name: np.asarray(values, dtype=float) for name, values in train_errs.items()
        }
        preds = np.concatenate(preds)  # pooled out-of-fold predictions
        trues = np.concatenate(trues)
        ss_res = float(np.sum((trues - preds) ** 2))
        ss_tot = float(np.sum((trues - trues.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        metric_stats = {}
        for loss_name in loss_fns:
            metric_stats[f"{loss_name}_mean"] = float(errs[loss_name].mean())
            metric_stats[f"{loss_name}_std"] = float(errs[loss_name].std())
            metric_stats[f"train_{loss_name}_mean"] = float(
                train_errs[loss_name].mean()
            )
            metric_stats[f"train_{loss_name}_std"] = float(train_errs[loss_name].std())
        rows.append(
            {
                **params,
                **metric_stats,
                "cv_mean": metric_stats[primary_mean_col],
                "cv_std": metric_stats[primary_std_col],
                "train_cv_mean": metric_stats[f"train_{primary_mean_col}"],
                "train_cv_std": metric_stats[f"train_{primary_std_col}"],
                "r2": r2,
                "pred_mean": float(preds.mean()),
                "pred_std": float(preds.std()),
                "true_mean": float(trues.mean()),
                "true_std": float(trues.std()),
                "_params": params,
                "_errs": errs,
                "_train_errs": train_errs,
            }
        )

    rows.sort(key=lambda r: r[primary_mean_col])
    best = rows[0]
    best_model = clone(model).set_params(**best["_params"]).fit(Xa, ya)
    results = pl.DataFrame(
        [{k: v for k, v in r.items() if not k.startswith("_")} for r in rows]
    )
    out = {
        "best_params": best["_params"],
        "best_model": best_model,
        "losses": list(loss_fns),
        "primary_loss": primary_loss,
        "cv_mean": best[primary_mean_col],
        "cv_std": best[primary_std_col],
        "train_cv_mean": best[f"train_{primary_mean_col}"],
        "train_cv_std": best[f"train_{primary_std_col}"],
        "r2": best["r2"],
        "pred_mean": best["pred_mean"],
        "pred_std": best["pred_std"],
        "true_mean": best["true_mean"],
        "true_std": best["true_std"],
        "fold_errors": best["_errs"][primary_loss],
        "train_fold_errors": best["_train_errs"][primary_loss],
        "results": results,
    }
    for loss_name in loss_fns:
        out[f"{loss_name}_mean"] = best[f"{loss_name}_mean"]
        out[f"{loss_name}_std"] = best[f"{loss_name}_std"]
        out[f"train_{loss_name}_mean"] = best[f"train_{loss_name}_mean"]
        out[f"train_{loss_name}_std"] = best[f"train_{loss_name}_std"]
        out[f"{loss_name}_fold_errors"] = best["_errs"][loss_name]
        out[f"train_{loss_name}_fold_errors"] = best["_train_errs"][loss_name]
    return out

### single feature checking

In [ ]:
# Market-level features: cross-stock arithmetic mean of log RV at trailing
# windows. mkt_logavg{w} = mean_x[ trailing-w arithmetic mean of log RV_x ].
# w=1 collapses to the spot log RV. These represent a "market" vol level and
# are added only to the single-feature correlation comparison below, not to
# the modelling specs in the next cell.
MKT_WINDOWS = [1, 5, 11, 15, 21]
MKT_FEATURES = [f"mkt_logavg{w}" for w in MKT_WINDOWS]


def _build_market_log_avg_features(vols_frame, stock_cols, windows, day_col="day"):
    sorted_vols = vols_frame.sort(day_col)
    exprs = []
    for w in windows:
        per_stock = [pl.col(f"{c}_avg{w}") if w > 1 else pl.col(c) for c in stock_cols]
        exprs.append(pl.mean_horizontal(per_stock).alias(f"mkt_logavg{w}"))
    return sorted_vols.select(day_col, *exprs)


_mkt_features_df = _build_market_log_avg_features(X, returns, MKT_WINDOWS)
# Idempotent: drop existing mkt columns from X before re-joining.
_existing_mkt = [c for c in MKT_FEATURES if c in X.columns]
if _existing_mkt:
    X = X.drop(_existing_mkt)
X = X.join(_mkt_features_df, on="day", how="left")


def market_level_candidates():
    return [(f"MKT log-avg{w}", f"mkt_logavg{w}") for w in MKT_WINDOWS]


def one_feature_candidates_for_stock(s, include_garch=True):
    candidates = [
        ("RV", s),
        ("avg5", f"{s}_avg5"),
        ("avg10", f"{s}_avg10"),
        ("avg11", f"{s}_avg11"),
        ("avg15", f"{s}_avg15"),
        ("avg21", f"{s}_avg21"),
        ("EWMA-1", f"{s}_ewma1"),
        ("EWMA-5", f"{s}_ewma5"),
        ("EWMA-10", f"{s}_ewma10"),
        ("EWMA-11", f"{s}_ewma11"),
        ("EWMA-15", f"{s}_ewma15"),
        ("EWMA-21", f"{s}_ewma21"),
        ("EWMA-44", f"{s}_ewma44"),
    ]
    if include_garch:
        candidates.append(("GARCH fwd", f"{s}_garch_fwd"))
    return [(label, feat) for label, feat in candidates if feat in X.columns]


def cross_stock_one_feature_candidates(
    source_stocks=returns, include_garch=True, include_market=True
):
    candidates = []
    for source_stock in source_stocks:
        for feature_label, feature in one_feature_candidates_for_stock(
            source_stock, include_garch=include_garch
        ):
            candidates.append((source_stock, feature_label, feature))
    if include_market:
        for feature_label, feature in market_level_candidates():
            if feature in X.columns:
                candidates.append(("MKT", feature_label, feature))
    return candidates


def _filter_day_range(frame, start_day=None, end_day=None, day_col="day"):
    out = frame
    if start_day is not None:
        out = out.filter(pl.col(day_col) >= start_day)
    if end_day is not None:
        out = out.filter(pl.col(day_col) <= end_day)
    return out


FULL_CV = globals().get(
    "FULL_CV", {"n_splits": 5, "purge": 21, "min_train": 63, "train_span": None}
)
POST_SHIFT_START_DAY = globals().get("POST_SHIFT_START_DAY", 151)
POST_SHIFT_CV = globals().get(
    "POST_SHIFT_CV", {"n_splits": 4, "purge": 21, "min_train": 63, "train_span": None}
)


def _display_target_stock_tables(
    frame,
    stocks_to_eval=returns,
    stock_col="stock",
    table_label="Target stock",
    tbl_cols=32,
    tbl_rows=80,
):
    with pl.Config(tbl_cols=tbl_cols, tbl_rows=tbl_rows):
        for stock in stocks_to_eval:
            stock_frame = frame.filter(pl.col(stock_col) == stock)
            if stock_frame.height == 0:
                continue
            print(f"{table_label}: {stock}")
            display(stock_frame)


def compare_one_feature_models(
    label,
    X_frame,
    y_frame,
    stocks_to_eval=returns,
    source_stocks=returns,
    include_garch=True,
    start_day=None,
    end_day=None,
    day_col="day",
    display_rows=100,
    **_ignored,  # absorb legacy CV kwargs (n_splits, purge, min_train, ...)
):
    """Rank single features by Pearson r(feature_t, target_fwd) per target stock.

    Features are info<=t and target is the forward window t+1..t+21, so a
    positive r means the feature carries directional information for forecasting
    future realized vol.
    """
    X_eval = _filter_day_range(X_frame, start_day, end_day)
    candidate_features = [
        (source_stock, feature_label, feature)
        for source_stock, feature_label, feature in cross_stock_one_feature_candidates(
            source_stocks, include_garch=include_garch
        )
        if feature in X_eval.columns
    ]

    rows = []
    for s in stocks_to_eval:
        target = f"{s}_fwd"
        for source_stock, feature_label, feature in candidate_features:
            joined = (
                X_eval.select(day_col, feature)
                .join(y_frame.select(day_col, target), on=day_col)
                .drop_nulls()
                .drop_nans()
            )
            n = joined.height
            if n < 3:
                pearson = None
            else:
                arr_f = joined[feature].to_numpy()
                arr_t = joined[target].to_numpy()
                if np.std(arr_f) == 0 or np.std(arr_t) == 0:
                    pearson = None
                else:
                    pearson = float(np.corrcoef(arr_f, arr_t)[0, 1])
            rows.append(
                {
                    "stock": s,
                    "source_stock": source_stock,
                    "same_stock": source_stock == s,
                    "feature_label": feature_label,
                    "feature": feature,
                    "n": n,
                    "pearson": round(pearson, 4) if pearson is not None else None,
                    "abs_pearson": (
                        round(abs(pearson), 4) if pearson is not None else None
                    ),
                }
            )

    comp = (
        pl.DataFrame(rows)
        .sort(["stock", "pearson"], descending=[False, True], nulls_last=True)
        .with_columns(
            pl.col("pearson")
            .rank(method="ordinal", descending=True)
            .over("stock")
            .cast(pl.Int64)
            .alias("rank_corr")
        )
    )

    summary = (
        comp.group_by(["source_stock", "feature_label"], maintain_order=True)
        .agg(
            pl.col("rank_corr").mean().round(2).alias("mean_rank_corr"),
            pl.col("pearson").mean().round(3).alias("mean_pearson"),
            pl.col("abs_pearson").mean().round(3).alias("mean_abs_pearson"),
        )
        .sort("mean_pearson", descending=True, nulls_last=True)
    )

    print(label)
    print(
        f"Candidate features per target: {len(candidate_features)}; rows: {comp.height}"
    )
    print(
        "Pearson r(feature at t, target forward RV over t+1..t+21). Higher is better; r<=0 means no usable signal."
    )
    print("Rows are ranked within each target stock by Pearson r (descending).")
    _display_target_stock_tables(
        comp,
        stocks_to_eval=stocks_to_eval,
        tbl_cols=12,
        tbl_rows=display_rows,
    )
    print("Averaged over target stocks, ranked by mean Pearson r:")
    display(summary)
    return comp, summary


one_feature_full, one_feature_full_summary = compare_one_feature_models(
    "Full-year one-feature correlation",
    X,
    y,
    returns,
)

one_feature_post, one_feature_post_summary = compare_one_feature_models(
    f"Post-shift one-feature correlation, day >= {POST_SHIFT_START_DAY}",
    X,
    y,
    returns,
    start_day=POST_SHIFT_START_DAY,
)


def compare_avg11_vs_ewma11_pass(comp, label):
    own_stock = comp.filter(
        pl.col("same_stock") & pl.col("feature_label").is_in(["avg11", "EWMA-11"])
    )
    wide = (
        own_stock.select("stock", "feature_label", "pearson", "abs_pearson")
        .pivot(values=["pearson", "abs_pearson"], index="stock", on="feature_label")
        .with_columns(
            (pl.col("pearson_EWMA-11") - pl.col("pearson_avg11"))
            .round(4)
            .alias("ewma11_minus_avg11_pearson"),
            pl.when(pl.col("pearson_EWMA-11") >= pl.col("pearson_avg11"))
            .then(pl.lit("EWMA-11"))
            .otherwise(pl.lit("avg11"))
            .alias("winner"),
        )
        .select(
            "stock",
            "pearson_avg11",
            "pearson_EWMA-11",
            "ewma11_minus_avg11_pearson",
            "winner",
        )
    )
    summary = own_stock.group_by("feature_label", maintain_order=True).agg(
        pl.col("pearson").mean().round(4),
        pl.col("abs_pearson").mean().round(4),
    )
    print(label)
    display(wide)
    print("Average same-stock Pearson:")
    display(summary)
    return wide, summary


avg11_vs_ewma11_full, avg11_vs_ewma11_full_summary = compare_avg11_vs_ewma11_pass(
    one_feature_full,
    "Full-year same-stock Pearson: avg11 vs EWMA-11",
)

avg11_vs_ewma11_post, avg11_vs_ewma11_post_summary = compare_avg11_vs_ewma11_pass(
    one_feature_post,
    f"Post-shift same-stock Pearson: avg11 vs EWMA-11, day >= {POST_SHIFT_START_DAY}",
)


def _feature_window_description(feature_label):
    if feature_label == "RV":
        return "daily RV at t"
    if feature_label.startswith("avg"):
        return f"trailing {feature_label[3:]} trading-day avg RV through t"
    if feature_label.startswith("EWMA-"):
        return f"EWMA half-life {feature_label.split('-')[-1]} trading days through t"
    if feature_label == "GARCH fwd":
        return "GARCH forward vol forecast from info through t"
    return feature_label


def identify_lead_lag_from_top_features(
    comp,
    label,
    top_n=5,
    stocks_to_eval=returns,
    cross_stock_only=True,
):
    frame = comp.filter(~pl.col("same_stock")) if cross_stock_only else comp
    top = (
        frame.sort(["stock", "rank_corr"])
        .group_by("stock", maintain_order=True)
        .head(top_n)
        .with_columns(
            pl.format("{} -> {}", pl.col("source_stock"), pl.col("stock")).alias(
                "lead_lag_edge"
            ),
            pl.col("feature_label")
            .map_elements(_feature_window_description, return_dtype=pl.String)
            .alias("source_window"),
            pl.when(pl.col("pearson") > 0)
            .then(pl.lit("positive"))
            .when(pl.col("pearson") < 0)
            .then(pl.lit("negative"))
            .otherwise(pl.lit("zero"))
            .alias("relation_sign"),
        )
        .select(
            "stock",
            "rank_corr",
            "lead_lag_edge",
            "source_stock",
            "feature_label",
            "feature",
            "source_window",
            "pearson",
            "abs_pearson",
            "relation_sign",
        )
    )
    edges = (
        top.group_by(["source_stock", "stock", "lead_lag_edge"], maintain_order=True)
        .agg(
            pl.len().alias("top5_feature_count"),
            pl.col("rank_corr").mean().round(2).alias("mean_rank"),
            pl.col("pearson").mean().round(3).alias("mean_pearson"),
            pl.col("feature_label").str.join(", ").alias("top_features"),
            pl.col("relation_sign").str.join(", ").alias("signs"),
        )
        .sort(
            ["top5_feature_count", "mean_rank", "mean_pearson"],
            descending=[True, False, True],
        )
    )
    source_summary = (
        top.group_by("source_stock", maintain_order=True)
        .agg(
            pl.len().alias("top5_appearances"),
            pl.n_unique("stock").alias("n_targets_led"),
            pl.col("pearson").mean().round(3).alias("mean_pearson"),
            pl.col("stock").unique().str.join(", ").alias("targets_led"),
        )
        .sort(
            ["top5_appearances", "n_targets_led", "mean_pearson"],
            descending=[True, True, True],
        )
    )
    print(label)
    print(
        f"Top {top_n} cross-stock features per target, ranked by Pearson r(feature_t, target_fwd). Edge means source feature at t correlates with target fwd RV over t+1..t+21."
    )
    _display_target_stock_tables(
        top, stocks_to_eval=stocks_to_eval, tbl_cols=14, tbl_rows=top_n
    )
    print("Repeated directed lead-lag edges among those top features:")
    display(edges)
    print("Leading source stocks among top features:")
    display(source_summary)
    return top, edges, source_summary


lead_lag_full, lead_lag_full_edges, lead_lag_full_sources = (
    identify_lead_lag_from_top_features(
        one_feature_full,
        "Full-year lead-lag from top one-feature correlations",
    )
)

lead_lag_post, lead_lag_post_edges, lead_lag_post_sources = (
    identify_lead_lag_from_top_features(
        one_feature_post,
        f"Post-shift lead-lag from top one-feature correlations, day >= {POST_SHIFT_START_DAY}",
    )
)

### modeling

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

har_base_cols = sorted(
    {
        c
        for s in returns
        for c in [
            s,
            f"{s}_avg5",
            f"{s}_avg10",
            f"{s}_avg11",
            f"{s}_avg15",
            f"{s}_avg21",
            f"{s}_ewma1",
            f"{s}_ewma5",
            f"{s}_ewma10",
            f"{s}_ewma11",
            f"{s}_ewma15",
            f"{s}_ewma21",
            f"{s}_ewma44",
        ]
    }
)
X = X.with_columns(
    [
        pl.when(pl.col(c) > 0).then(pl.col(c).log()).otherwise(None).alias(f"log_{c}")
        for c in har_base_cols
    ]
)

# Common rows so every model in a date range is scored on the same purged CV windows.
Xc = X.drop_nulls().drop_nans()
ALL_STOCK_AVG_EWMA_FEATURES = [c for c in har_base_cols if c in X.columns]
ALL_STOCK_LOG_AVG_EWMA_FEATURES = [
    f"log_{c}" for c in ALL_STOCK_AVG_EWMA_FEATURES if f"log_{c}" in X.columns
]

ALPHA_GRID = [0.01, 0.1, 1.0, 10.0]
L1_RATIO_GRID = [0.1, 0.5, 0.9]
STAGEWISE_MAX_FEATURES = 8
STAGEWISE_MIN_COEF = 0.0
FULL_CV = {"n_splits": 5, "purge": 21, "min_train": 63, "train_span": None}
POST_SHIFT_START_DAY = 151
POST_SHIFT_CV = {"n_splits": 4, "purge": 21, "min_train": 63, "train_span": None}
REPORT_RANK_METRIC = "qlike"


def stagewise_features_for_stock(s):
    return ALL_STOCK_AVG_EWMA_FEATURES


def model_specs_for_stock(s):
    har_feats = [s, f"{s}_avg5", f"{s}_avg21"]
    har_ewma11_feats = [*har_feats, f"{s}_ewma11"]
    ewma_har_feats = [f"{s}_ewma1", f"{s}_ewma5", f"{s}_ewma21"]
    log_har_feats = [f"log_{c}" for c in har_feats]
    log_ewma_har_feats = [f"log_{c}" for c in ewma_har_feats]
    all_stock_log_avg_ewma_feats = ALL_STOCK_LOG_AVG_EWMA_FEATURES
    stagewise_feats = stagewise_features_for_stock(s)
    specs = [
        ("HAR (LinReg)", LinearRegression(), har_feats, None),
        ("HAR (Ridge 0.1)", Ridge(alpha=0.1), har_feats, None),
        ("HAR (Lasso 0.1)", Lasso(alpha=0.1, max_iter=10000), har_feats, None),
        (
            "HAR (Ridge scaled)",
            make_pipeline(StandardScaler(), Ridge()),
            har_feats,
            {"ridge__alpha": ALPHA_GRID},
        ),
        (
            "HAR+EWMA-11 (Ridge scaled)",
            make_pipeline(StandardScaler(), Ridge()),
            har_ewma11_feats,
            {"ridge__alpha": ALPHA_GRID},
        ),
        ("EWMA-HAR (LinReg)", LinearRegression(), ewma_har_feats, None),
        (
            "EWMA-HAR (Ridge scaled)",
            make_pipeline(StandardScaler(), Ridge()),
            ewma_har_feats,
            {"ridge__alpha": ALPHA_GRID},
        ),
        (
            "EWMA-HAR (Lasso scaled)",
            make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
            ewma_har_feats,
            {"lasso__alpha": ALPHA_GRID},
        ),
        (
            "EWMA-HAR (ElasticNet scaled)",
            make_pipeline(StandardScaler(), ElasticNet(max_iter=10000)),
            ewma_har_feats,
            {"elasticnet__alpha": ALPHA_GRID, "elasticnet__l1_ratio": L1_RATIO_GRID},
        ),
        (
            "Forward stagewise all-stock (LinReg)",
            ForwardStagewiseRegressor(
                max_features=STAGEWISE_MAX_FEATURES,
                min_coef=STAGEWISE_MIN_COEF,
                feature_names=stagewise_feats,
            ),
            stagewise_feats,
            None,
        ),
        (
            "HAR (Lasso scaled)",
            make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
            har_feats,
            {"lasso__alpha": ALPHA_GRID},
        ),
        (
            "HAR (ElasticNet scaled)",
            make_pipeline(StandardScaler(), ElasticNet(max_iter=10000)),
            har_feats,
            {"elasticnet__alpha": ALPHA_GRID, "elasticnet__l1_ratio": L1_RATIO_GRID},
        ),
        (
            "log-HAR",
            TransformedTargetRegressor(
                regressor=LinearRegression(),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_har_feats,
            None,
        ),
        (
            "log-HAR (Ridge scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), Ridge()),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_har_feats,
            {"regressor__ridge__alpha": ALPHA_GRID},
        ),
        (
            "log-HAR (Lasso scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_har_feats,
            {"regressor__lasso__alpha": ALPHA_GRID},
        ),
        (
            "log-HAR (ElasticNet scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), ElasticNet(max_iter=10000)),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_har_feats,
            {
                "regressor__elasticnet__alpha": ALPHA_GRID,
                "regressor__elasticnet__l1_ratio": L1_RATIO_GRID,
            },
        ),
        (
            "log-EWMA-HAR",
            TransformedTargetRegressor(
                regressor=LinearRegression(),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_ewma_har_feats,
            None,
        ),
        (
            "log-EWMA-HAR (Ridge scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), Ridge()),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_ewma_har_feats,
            {"regressor__ridge__alpha": ALPHA_GRID},
        ),
        (
            "log-EWMA-HAR (Lasso scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_ewma_har_feats,
            {"regressor__lasso__alpha": ALPHA_GRID},
        ),
        (
            "log-EWMA-HAR (ElasticNet scaled)",
            TransformedTargetRegressor(
                regressor=make_pipeline(StandardScaler(), ElasticNet(max_iter=10000)),
                func=np.log,
                inverse_func=np.exp,
                check_inverse=False,
            ),
            log_ewma_har_feats,
            {
                "regressor__elasticnet__alpha": ALPHA_GRID,
                "regressor__elasticnet__l1_ratio": L1_RATIO_GRID,
            },
        ),
        (
            "All-stock log avg+EWMA (ElasticNet scaled)",
            make_pipeline(StandardScaler(), ElasticNet(max_iter=20000)),
            all_stock_log_avg_ewma_feats,
            {"elasticnet__alpha": ALPHA_GRID, "elasticnet__l1_ratio": L1_RATIO_GRID},
        ),
    ]
    baselines = []
    if f"{s}_ewma11" in X.columns:
        baselines.append(
            ("EWMA-11 (pass)", PassthroughRegressor(), [f"{s}_ewma11"], None)
        )
    if f"{s}_garch_fwd" in X.columns:
        baselines.append(
            ("GARCH (pass)", PassthroughRegressor(), [f"{s}_garch_fwd"], None)
        )
    return [
        spec
        for spec in [*specs, *baselines]
        if spec[2] and all(feature in X.columns for feature in spec[2])
    ]


def _best_str(p):
    if not p:
        return "-"
    # strip sklearn pipeline prefix (e.g. "ridge__alpha" -> "alpha")
    return ", ".join(f"{k.split('__')[-1]}={v:g}" for k, v in p.items())


def _fold_str(errors):
    return "[" + ", ".join(f"{e:.2f}" for e in errors) + "]"


def _selected_features_str(model):
    features = getattr(model, "selected_feature_names_", None)
    return ", ".join(features) if features else "-"


def _first_selected_feature(model):
    features = getattr(model, "selected_feature_names_", None)
    return features[0] if features else "-"


def _stagewise_coef_str(model):
    selected = getattr(model, "selected_idx_", None)
    coef = getattr(model, "coef_", None)
    names = getattr(model, "feature_names", None)
    if selected is None or coef is None or names is None or len(selected) == 0:
        return "-"
    return ", ".join(f"{names[i]}={coef[i]:.3g}" for i in selected)


def _stagewise_intercept_str(model):
    if not hasattr(model, "selected_idx_"):
        return "-"
    intercept = getattr(model, "intercept_", None)
    return "-" if intercept is None else f"{intercept:.3g}"


def _fold_table(X_eval, y_frame, cv, day_col="day"):
    eval_frame = (
        X_eval.select(day_col)
        .join(y_frame, on=day_col)
        .sort(day_col)
        .drop_nulls()
        .drop_nans()
    )
    days = eval_frame[day_col].to_numpy()
    rows = []
    for i, (tr, te) in enumerate(cv.split(eval_frame), start=1):
        rows.append(
            {
                "fold": i,
                "train_days": f"{days[tr[0]]}-{days[tr[-1]]}",
                "test_days": f"{days[te[0]]}-{days[te[-1]]}",
                "n_train": len(tr),
                "n_test": len(te),
            }
        )
    return pl.DataFrame(rows), eval_frame


def _report_row(stock, name, feats, r):
    row = {
        "stock": stock,
        "model": name,
        "features": ", ".join(feats),
        "primary_loss": r.get("primary_loss", "rmse"),
    }
    for loss_name in r.get("losses", ["rmse"]):
        row[f"val_{loss_name}"] = round(r[f"{loss_name}_mean"], 3)
        row[f"train_{loss_name}"] = round(r[f"train_{loss_name}_mean"], 3)
        row[f"val_{loss_name}_std"] = round(r[f"{loss_name}_std"], 3)
        row[f"train_{loss_name}_std"] = round(r[f"train_{loss_name}_std"], 3)
    row.update(
        {
            "train_folds": _fold_str(r["train_fold_errors"]),
            "val_folds": _fold_str(r["fold_errors"]),
            "r2": round(r["r2"], 3),
            "pred_mean": round(r["pred_mean"], 3),
            "pred_std": round(r["pred_std"], 3),
            "true_mean": round(r["true_mean"], 3),
            "true_std": round(r["true_std"], 3),
            "best": _best_str(r["best_params"]),
            "first_feature": _first_selected_feature(r["best_model"]),
            "selected_features": _selected_features_str(r["best_model"]),
            "stagewise_intercept": _stagewise_intercept_str(r["best_model"]),
            "stagewise_coefs": _stagewise_coef_str(r["best_model"]),
        }
    )
    return row


def summarize_report(report, prefix=None):
    metric_cols = [
        c
        for c in report.columns
        if (c.startswith("val_") or c.startswith("train_"))
        and c not in {"val_folds", "train_folds"}
    ]
    exprs = [pl.col(c).mean().round(3).alias(c) for c in metric_cols]
    exprs.append(pl.col("r2").mean().round(3).alias("r2"))
    summary = report.group_by("model", maintain_order=True).agg(exprs)
    if prefix is not None:
        summary = summary.rename(
            {c: f"{prefix}_{c}" for c in summary.columns if c != "model"}
        )
    return summary


def _display_target_stock_tables(
    frame,
    stocks_to_eval=returns,
    stock_col="stock",
    table_label="Target stock",
    tbl_cols=32,
    tbl_rows=80,
):
    with pl.Config(tbl_cols=tbl_cols, tbl_rows=tbl_rows):
        for stock in stocks_to_eval:
            stock_frame = frame.filter(pl.col(stock_col) == stock)
            if stock_frame.height == 0:
                continue
            print(f"{table_label}: {stock}")
            display(stock_frame)


def run_model_report(
    label,
    X_frame,
    y_frame,
    stocks_to_eval=returns,
    start_day=None,
    end_day=None,
    n_splits=5,
    purge=21,
    min_train=63,
    train_span=None,
    horizon=21,
    loss_fns=None,
    primary_loss="rmse",
    rank_metric=REPORT_RANK_METRIC,
    display_rows=80,
):
    cv = PurgedWalkForwardCV(
        n_splits=n_splits,
        purge=purge,
        min_train=min_train,
        train_span=train_span,
    )
    X_eval = _filter_day_range(X_frame, start_day, end_day).drop_nulls().drop_nans()
    folds, eval_frame = _fold_table(X_eval, y_frame, cv)

    rows = []
    for s in stocks_to_eval:
        for name, model, feats, grid in model_specs_for_stock(s):
            r = evaluate_model(
                X_eval,
                y_frame,
                model,
                feats,
                f"{s}_fwd",
                param_grid=grid,
                cv=cv,
                horizon=horizon,
                loss_fns=loss_fns,
                primary_loss=primary_loss,
            )
            rows.append(_report_row(s, name, feats, r))

    report = pl.DataFrame(rows)
    sort_loss_col = f"val_{rank_metric}"
    if sort_loss_col not in report.columns:
        sort_loss_col = (
            f"val_{primary_loss}"
            if f"val_{primary_loss}" in report.columns
            else "val_rmse"
        )
    sort_std_col = f"{sort_loss_col}_std"
    sort_cols = ["stock", sort_loss_col]
    if sort_std_col in report.columns:
        sort_cols.append(sort_std_col)
    report = report.sort(sort_cols)
    summary = summarize_report(report)
    if sort_loss_col in summary.columns:
        summary = summary.sort(sort_loss_col)
    print(label)
    print(f"Rows: X={X_eval.shape}, eval={eval_frame.shape}")
    display(folds)
    print(
        "Loss columns show train/validation means. Fold columns use the primary loss per split. Pred / true stats are on the pooled OOF set."
    )
    print(f"Rows are ranked within stock by {sort_loss_col} and then {sort_std_col}.")
    _display_target_stock_tables(
        report,
        stocks_to_eval=stocks_to_eval,
        tbl_cols=32,
        tbl_rows=display_rows,
    )
    print(f"Averaged over the {len(stocks_to_eval)} stocks:")
    with pl.Config(tbl_cols=32):
        display(summary)
    return {
        "label": label,
        "X": X_eval,
        "cv": cv,
        "folds": folds,
        "report": report,
        "summary": summary,
    }


full_eval = run_model_report("Full-year CV", X, y, returns, **FULL_CV)
report = full_eval["report"]
full_summary = summarize_report(report, prefix="full")

In [ ]:
post_shift_eval = run_model_report(
    f"Post-shift CV, day >= {POST_SHIFT_START_DAY}",
    X,
    y,
    returns,
    start_day=POST_SHIFT_START_DAY,
    **POST_SHIFT_CV,
)
post_report = post_shift_eval["report"]
post_summary = summarize_report(post_report, prefix="post")

comparison = (
    full_summary.join(post_summary, on="model")
    .with_columns(
        (pl.col("post_val_rmse") - pl.col("full_val_rmse")).alias(
            "post_minus_full_rmse"
        ),
        (pl.col("post_val_qlike") - pl.col("full_val_qlike")).alias(
            "post_minus_full_qlike"
        ),
    )
    .sort("post_val_qlike")
)
display(comparison)

post_top3 = (
    post_report.sort(["stock", "val_qlike", "val_qlike_std"])
    .group_by("stock", maintain_order=True)
    .head(3)
    .select(
        "stock",
        "model",
        "features",
        "val_qlike",
        "train_qlike",
        "val_qlike_std",
        "val_rmse",
        "train_rmse",
        "val_mse",
        "train_mse",
        "val_rmse_std",
        "r2",
        "pred_mean",
        "true_mean",
        "best",
    )
)
print("Post-shift top 3 models per target stock:")
_display_target_stock_tables(post_top3, stocks_to_eval=returns, tbl_cols=32, tbl_rows=3)